# Aplikasi Data Mining Clustering Segmentasi Konsumen (K-Means + Streamlit + Cloudflare Tunnel)
**Penting:** Sebelum menjalankan sel-sel di bawah ini, pastikan Anda sudah mengunggah file **`Mall_Customers.csv`** ke menu **Files** (ikon folder di sebelah kiri Google Colab).

In [ ]:
# 1. Install library yang dibutuhkan & Download Cloudflare Tunnel
!pip install -q streamlit scikit-learn pandas matplotlib seaborn openpyxl
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print('Instalasi dan persiapan Cloudflare Tunnel selesai!')

## Bagian 1: Preparation Model & Clustering di Notebook (Video 1)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

# Membaca Dataset
df = pd.read_csv('Mall_Customers.csv')
print('5 Data Pertama:')
display(df.head())

# Persiapan Dataset (Rename & Drop Kolom)
df.rename(index=str, columns={
    'Annual Income (k$)': 'Income',
    'Spending Score (1-100)': 'Score'
}, inplace=True)

X = df.drop(['CustomerID', 'Gender'], axis=1)
print('\nDataset setelah persiapan (X):')
display(X.head())

# Menentukan Jumlah Cluster dengan Elbow
clusters = []
for i in range(1, 11):
    km = KMeans(n_clusters=i, n_init=10, random_state=42).fit(X)
    clusters.append(km.inertia_)

fig, ax = plt.subplots(figsize=(12, 8))
sns.lineplot(x=list(range(1, 11)), y=clusters, ax=ax)
ax.set_title('Mencari Elbow')
ax.set_xlabel('Clusters')
ax.set_ylabel('Inertia')

# Panah Elbow
ax.annotate('Possible elbow point', xy=(3, 140000), xytext=(3, 50000), xycoords='data',
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3', color='blue', lw=2))
ax.annotate('Possible elbow point', xy=(5, 80000), xytext=(5, 150000), xycoords='data',
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3', color='blue', lw=2))
plt.show()

# Proses Clustering dengan K=4
n_clust = 4
kmean = KMeans(n_clusters=n_clust, n_init=10, random_state=42).fit(X)
X['Labels'] = kmean.labels_

plt.figure(figsize=(10, 8))
sns.scatterplot(x=X['Income'], y=X['Score'], hue=X['Labels'], markers=True,
                size=X['Labels'], palette=sns.color_palette('hls', n_clust))

for label in X['Labels'].unique():
    plt.annotate(label,
                 (X[X['Labels'] == label]['Income'].mean(),
                  X[X['Labels'] == label]['Score'].mean()),
                 horizontalalignment='center',
                 verticalalignment='center',
                 size=20, weight='bold',
                 color='black')
plt.show()

# Simpan ke Excel
X.to_excel('output_cluster.xlsx', index=False)
print('Hasil klaster berhasil disimpan ke output_cluster.xlsx')

## Bagian 2: Membuat File Aplikasi Streamlit `mall-customer.py` (Video 2)

In [ ]:
%%writefile mall-customer.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

df = pd.read_csv('Mall_Customers.csv')

df.rename(index=str, columns={
    'Annual Income (k$)': 'Income',
    'Spending Score (1-100)': 'Score'
}, inplace=True)

X = df.drop(['CustomerID', 'Gender'], axis=1)

st.header("Isi Dataset")
st.write(X)

# Menampilkan Panah Elbow
clusters = []
for i in range(1, 11):
    km = KMeans(n_clusters=i, n_init=10, random_state=42).fit(X)
    clusters.append(km.inertia_)

fig, ax = plt.subplots(figsize=(12, 8))
sns.lineplot(x=list(range(1, 11)), y=clusters, ax=ax)
ax.set_title('Mencari Elbow')
ax.set_xlabel('Clusters')
ax.set_ylabel('Inertia')

ax.annotate('Possible elbow point', xy=(3, 140000), xytext=(3, 50000), xycoords='data',
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3', color='blue', lw=2))
ax.annotate('Possible elbow point', xy=(5, 80000), xytext=(5, 150000), xycoords='data',
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3', color='blue', lw=2))

st.header("Elbow Point")
st.pyplot(fig)

# Sidebar jumlah K
st.sidebar.subheader("Nilai Jumlah K")
clust = st.sidebar.slider("Pilih jumlah cluster :", 2, 10, 3, 1)

def k_means(n_clust):
    kmean = KMeans(n_clusters=n_clust, n_init=10, random_state=42).fit(X)
    X['Labels'] = kmean.labels_

    fig_cluster = plt.figure(figsize=(10, 8))
    sns.scatterplot(x=X['Income'], y=X['Score'], hue=X['Labels'], markers=True,
                    size=X['Labels'], palette=sns.color_palette('hls', n_clust))

    for label in X['Labels'].unique():
        plt.annotate(label,
                     (X[X['Labels'] == label]['Income'].mean(),
                      X[X['Labels'] == label]['Score'].mean()),
                     horizontalalignment='center',
                     verticalalignment='center',
                     size=20, weight='bold',
                     color='black')

    st.header('Cluster Plot')
    st.pyplot(fig_cluster)
    st.write(X)

k_means(clust)

## Bagian 3: Menjalankan Streamlit via Cloudflare Tunnel
Jalankan sel di bawah ini, lalu klik tautan berakhiran **`.trycloudflare.com`** yang muncul pada output untuk membuka antarmuka web Streamlit.

In [ ]:
import subprocess
import time

streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "mall-customer.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)
print('Membuka Cloudflare Tunnel... Klik link .trycloudflare.com yang muncul di bawah ini:')
!./cloudflared tunnel --url http://localhost:8501 --no-autoupdate